## 1. Introduction

## 2. Data Acquisition

In [14]:
import requests
import pandas as pd
import json
import time
import os

In [78]:
import numpy as np
from pathlib import Path

data_path = Path.cwd().parent / "data"

work_bank_df = pd.read_csv(
    "https://ourworldindata.org/grapher/world-bank-income-groups.csv"
    "?v=1&csvType=full&useColumnShortNames=false",
    storage_options={"User-Agent": "Our World In Data data fetch/1.0"}
)

print("World Bank shape:", work_bank_df.shape)
print(work_bank_df.head())

WB_TO_ACLED = {
    "Congo":        "Republic of Congo",       # was "Congo, Rep."
    "Czechia":      "Czech Republic",
    "Eswatini":     "eSwatini",
    "Congo, Dem. Rep.":  "Democratic Republic of Congo",
    "Côte d'Ivoire":     "Ivory Coast",
    "Korea, Rep.":       "South Korea",
    "Korea, Dem. People's Rep.": "North Korea",
    "Lao PDR":           "Laos",
    "Syrian Arab Republic": "Syria",
    "Yemen, Rep.":       "Yemen",
    "Venezuela, RB":     "Venezuela",
    "Iran, Islamic Rep.": "Iran",
    "Egypt, Arab Rep.":  "Egypt",
    "Kyrgyz Republic":   "Kyrgyzstan",
    "Turkiye":           "Turkey",
}

work_bank_df["Entity"] = work_bank_df["Entity"].replace(WB_TO_ACLED)

INCOME_COL = "World Bank's income classification"

# Keep only the most recent classification per country
wb_latest = (
    work_bank_df
    .sort_values("Year", ascending=False)
    .drop_duplicates(subset="Entity")
)

global_south = wb_latest.loc[
    wb_latest[INCOME_COL] != 'High-income countries',
    'Entity'
].unique().tolist()

western = wb_latest.loc[
    wb_latest[INCOME_COL] == 'High-income countries',
    'Entity'
].unique().tolist()

# Manual overrides for geopolitically misclassified countries
OVERRIDES_WESTERN = ["Ukraine"]
OVERRIDES_GLOBAL_SOUTH = [
    "Saudi Arabia", "Qatar", "Kuwait",
    "Bahrain", "Oman", "United Arab Emirates"
]

for c in OVERRIDES_WESTERN:
    if c not in western:
        western.append(c)
    if c in global_south:
        global_south.remove(c)

for c in OVERRIDES_GLOBAL_SOUTH:
    if c not in global_south:
        global_south.append(c)
    if c in western:
        western.remove(c)

print(f"\nGlobal South countries: {len(global_south)}")
print(f"Western countries:      {len(western)}")

fatalities = pd.read_csv(
    f"{data_path}/number_of_reported_fatalities_by_country-year_as-of-03Apr2026.csv",
    skiprows=1
)

fatalities = fatalities.loc[:, ~fatalities.columns.str.contains('^Unnamed')]

print("\nFatalities shape:", fatalities.shape)
print(fatalities.head())

# Filter to study window 2015-2024 only
# ACLED has data from 1997 but coverage is sparse before 2015.
# We also exclude 2025 and 2026 which are outside our Guardian date range.
fatalities = fatalities[
    (fatalities['YEAR'] >= 2015) &
    (fatalities['YEAR'] <= 2024)
]

print(f"\nAfter date filter: {fatalities.shape}")
print(fatalities['YEAR'].value_counts().sort_index())

# =============================================================================
# ADD REGION CLASSIFICATION TO FATALITIES
# =============================================================================

fatalities["region_group"] = fatalities["COUNTRY"].apply(
    lambda x: "Western" if x in western else "Global South"
)

print("\nRegion group distribution:")
print(fatalities["region_group"].value_counts())

# Sanity check — countries not matched to either list
unmatched = fatalities[
    ~fatalities["COUNTRY"].isin(western + global_south)
]["COUNTRY"].unique()
print(f"\nUnmatched countries: {len(unmatched)}")
if len(unmatched) > 0:
    print(sorted(unmatched))

# =============================================================================
# SAVE CLEAN VERSION
# =============================================================================

fatalities.to_csv(f"{data_path}/cleaned_fatalities.csv", index=False)
print(f"\nSaved cleaned_fatalities.csv — shape: {fatalities.shape}")

World Bank shape: (7953, 4)
        Entity Code  Year World Bank's income classification
0  Afghanistan  AFG  1987               Low-income countries
1  Afghanistan  AFG  1988               Low-income countries
2  Afghanistan  AFG  1989               Low-income countries
3  Afghanistan  AFG  1990               Low-income countries
4  Afghanistan  AFG  1991               Low-income countries

Global South countries: 141
Western countries:      85

Fatalities shape: (2943, 3)
       COUNTRY  YEAR  FATALITIES
0  Afghanistan  2017       36360
1  Afghanistan  2018       42991
2  Afghanistan  2019       41419
3  Afghanistan  2020       31037
4  Afghanistan  2021       42425

After date filter: (1634, 3)
YEAR
2015     61
2016     74
2017     77
2018    161
2019    161
2020    212
2021    245
2022    215
2023    213
2024    215
Name: count, dtype: int64

Region group distribution:
region_group
Global South    1221
Western          413
Name: count, dtype: int64

Unmatched countries: 33
['Akroti

In [53]:
with open("../keys.json") as f:
    keys = json.load(f)

API_KEYS = [
    keys["guardian"]["api_key_1"],
    keys["guardian"]["api_key_2"],
    keys["guardian"]["api_key_3"]
]


In [55]:
acled = pd.read_csv("../data/cleaned_fatalities.csv")

print("=== ACLED Dataset ===")
print(f"Shape: {acled.shape}")
print(f"Columns: {list(acled.columns)}")
print(f"Countries: {acled['COUNTRY'].nunique()} unique")
print(f"Year range: {acled['YEAR'].min()} – {acled['YEAR'].max()}")
acled.head()

=== ACLED Dataset ===
Shape: (1634, 4)
Columns: ['COUNTRY', 'YEAR', 'FATALITIES', 'region_group']
Countries: 245 unique
Year range: 2015 – 2024


,COUNTRY,YEAR,FATALITIES,region_group
0,Afghanistan,2017,36360,Global South
1,Afghanistan,2018,42991,Global South
2,Afghanistan,2019,41419,Global South
3,Afghanistan,2020,31037,Global South
4,Afghanistan,2021,42425,Global South


In [56]:
FROM_DATE = "2015-01-01"
TO_DATE   = "2024-12-31"

# --- Automatic conversion: lowercase and replace spaces with hyphens ---
acled_countries = acled["COUNTRY"].unique().tolist()

def name_to_slug(name):
    return name.lower().strip().replace(" ", "-").replace("'", "").replace(",", "")

auto_slugs = {country: name_to_slug(country) for country in acled_countries}

In [57]:
SLUG_CORRECTIONS = {
    "Democratic Republic of Congo": "democratic-republic-of-congo",
    "Republic of Congo":            "republic-of-congo",
    "United States":                "us-news",
    "Myanmar":                      "myanmar",
    "Ivory Coast":                  "ivory-coast",
    "Burkina Faso":                 "burkina-faso",
    "South Sudan":                  "south-sudan",
    "Central African Republic":     "central-african-republic",
    "Bosnia-Herzegovina":           "bosnia-and-herzegovina",
    "North Korea":                  "north-korea",
    "South Korea":                  "south-korea",
    "Saudi Arabia":                 "saudi-arabia",
    "Sri Lanka":                    "sri-lanka",
    "Papua New Guinea":             "papua-new-guinea",
}

In [58]:
COUNTRY_TO_SLUG = {**auto_slugs, **SLUG_CORRECTIONS}

# Final list of (country_name, guardian_slug) pairs
COUNTRY_LIST = [(country, COUNTRY_TO_SLUG[country]) for country in acled_countries]

print(f"\nTotal countries to query: {len(COUNTRY_LIST)}")
print("\nSample mappings:")
for name, slug in COUNTRY_LIST[:5]:
    print(f"  {name} -> world/{slug}")


Total countries to query: 245

Sample mappings:
  Afghanistan -> world/afghanistan
  Akrotiri and Dhekelia -> world/akrotiri-and-dhekelia
  Albania -> world/albania
  Algeria -> world/algeria
  American Samoa -> world/american-samoa


In [20]:
BASE_URL    = "https://content.guardianapis.com/search"
current_key = 0  

def fetch_articles(country_name, slug):
    """
    Fetches all Guardian articles for a given country between FROM_DATE
    and TO_DATE. Paginates automatically and rotates API keys on rate
    limiting.

    For USA, UK, and Australia the Guardian uses dedicated top-level
    sections rather than world/ tags. These are detected by the "section:"
    prefix in the slug and queried using the section parameter instead of tag.

    Parameters:
        country_name : str — original ACLED country name, stored in output
        slug         : str — either "section:sectionid" or a world/ tag slug

    Returns a list of article dicts.
    """
    global current_key
    articles = []
    page = 1

    # Detect whether to query by section or by tag
    if slug.startswith("section:"):
        query_param = {"section": slug.replace("section:", "")}
    else:
        query_param = {"tag": f"world/{slug}"}

    while True:
        params = {
            **query_param,
            "from-date":   FROM_DATE,
            "to-date":     TO_DATE,
            "show-fields": "wordcount,sectionName",
            "page-size":   200,
            "page":        page,
            "api-key":     API_KEYS[current_key]
        }

        r = requests.get(BASE_URL, params=params)
        time.sleep(0.1)

        if r.status_code == 429:
            current_key = (current_key + 1) % len(API_KEYS)
            print(f"  Rate limited — switching to key {current_key}")
            continue

        if r.status_code != 200:
            print(f"  Error {r.status_code} for {slug} page {page} — skipping")
            break

        data    = r.json()["response"]
        results = data.get("results", [])

        if not results:
            break

        for a in results:
            articles.append({
                "country":   country_name,
                "date":      a["webPublicationDate"],
                "section":   a["sectionName"],
                "wordcount": a.get("fields", {}).get("wordcount", None),
                "title":     a["webTitle"],
                "pillar":    a.get("pillarName", None)
            })

        if page >= data["pages"]:
            break

        page += 1

    return articles


In [21]:
# --- Run collection (skips if file already exists) ---
RAW_PATH = "../data/guardian_raw.csv"

if os.path.exists(RAW_PATH) and os.path.getsize(RAW_PATH) > 0:
    print("Guardian raw data already collected — loading from file.")
    guardian_raw = pd.read_csv(RAW_PATH)
else:
    all_articles = []
    for country_name, slug in COUNTRY_LIST:
        print(f"Fetching: {country_name}...")
        articles = fetch_articles(country_name, slug)
        all_articles.extend(articles)
        print(f"  -> {len(articles)} articles")

    guardian_raw = pd.DataFrame(all_articles)
    guardian_raw.to_csv(RAW_PATH, index=False)
    print(f"\nSaved {len(guardian_raw)} total articles to {RAW_PATH}")

print(f"\nShape: {guardian_raw.shape}")
print(f"Countries with articles: {guardian_raw['country'].nunique()}")
print(f"Date range: {guardian_raw['date'].min()} to {guardian_raw['date'].max()}")
guardian_raw.head()

Guardian raw data already collected — loading from file.

Shape: (163282, 6)
Countries with articles: 194
Date range: 2015-01-01T01:13:56Z to 2024-12-31T22:07:54Z


,country,date,section,wordcount,title,pillar
0,Afghanistan,2024-12-31T18:41:27Z,UK news,495,Special forces troops face prosecution over al...,News
1,Afghanistan,2024-12-29T17:09:11Z,Media,471,David Page obituary,News
2,Afghanistan,2024-12-23T17:00:21Z,US news,2008,‘Never underestimate what neighbors can do for...,News
3,Afghanistan,2024-12-06T17:50:37Z,Global development,446,Taliban move to ban women training as nurses a...,News
4,Afghanistan,2024-12-06T10:40:29Z,Sport,471,Top Afghan cricketers urge Taliban to reverse ...,Sport


In [24]:
TOP_UP = [
    ("United States", "section:us-news"),
    ("United Kingdom", "section:uk-news"),
    ("Australia",      "section:australia-news"),
]

existing = pd.read_csv("../data/guardian_raw.csv")
existing = existing[~existing["country"].isin(["United States", "United Kingdom", "Australia"])]

top_up_articles = []
for country_name, slug in TOP_UP:
    print(f"Fetching: {country_name}...")
    articles = fetch_articles(country_name, slug)
    top_up_articles.extend(articles)
    print(f"  -> {len(articles)} articles")

top_up_df = pd.DataFrame(top_up_articles)
guardian_raw = pd.concat([existing, top_up_df], ignore_index=True)
guardian_raw.to_csv("../data/guardian_raw.csv", index=False)
print(f"Updated dataset: {len(guardian_raw)} total articles")

Fetching: United States...
  Error 400 for section:us-news page 191 — skipping
  -> 38000 articles
Fetching: United Kingdom...


KeyboardInterrupt: 

In [26]:
existing = pd.read_csv("../data/guardian_raw.csv")
existing = existing[~existing["country"].isin(["United States", "United Kingdom", "Australia"])]

partial_df = pd.DataFrame(top_up_articles)
guardian_raw = pd.concat([existing, partial_df], ignore_index=True)
guardian_raw.to_csv("../data/guardian_raw.csv", index=False)
print(f"Saved so far: {len(guardian_raw)} articles")
print(guardian_raw["country"].value_counts().tail(5))

Saved so far: 201282 articles
country
San Marino               9
Liechtenstein            7
Andorra                  5
Comoros                  5
Sao Tome and Principe    5
Name: count, dtype: int64


In [27]:
def fetch_articles_slow(country_name, slug):
    """Same as fetch_articles but with longer sleep for large sections."""
    global current_key
    articles = []
    page = 1

    if slug.startswith("section:"):
        query_param = {"section": slug.replace("section:", "")}
    else:
        query_param = {"tag": f"world/{slug}"}

    while True:
        params = {
            **query_param,
            "from-date":   FROM_DATE,
            "to-date":     TO_DATE,
            "show-fields": "wordcount,sectionName",
            "page-size":   200,
            "page":        page,
            "api-key":     API_KEYS[current_key]
        }

        try:
            r = requests.get(BASE_URL, params=params, timeout=30)
            time.sleep(0.5)  # slower — reduces stalling risk
        except requests.exceptions.Timeout:
            print(f"  Timeout on page {page} — retrying...")
            time.sleep(5)
            continue

        if r.status_code == 429:
            current_key = (current_key + 1) % len(API_KEYS)
            print(f"  Rate limited — switching to key {current_key}")
            time.sleep(2)
            continue

        if r.status_code != 200:
            print(f"  Error {r.status_code} on page {page} — stopping")
            break

        data    = r.json()["response"]
        results = data.get("results", [])
        if not results:
            break

        for a in results:
            articles.append({
                "country":   country_name,
                "date":      a["webPublicationDate"],
                "section":   a["sectionName"],
                "wordcount": a.get("fields", {}).get("wordcount", None),
                "title":     a["webTitle"],
                "pillar":    a.get("pillarName", None)
            })

        if page % 50 == 0:
            print(f"  ...page {page}, {len(articles)} articles so far")

        if page >= data["pages"]:
            break

        page += 1

    return articles

In [29]:
aus_articles = fetch_articles_slow("Australia", "section:australia-news")
print(f"Australia: {len(aus_articles)} articles")

  ...page 50, 10000 articles so far
  ...page 100, 20000 articles so far
  ...page 150, 30000 articles so far
  Error 400 on page 191 — stopping
Australia: 38000 articles


In [30]:
existing = pd.read_csv("../data/guardian_raw.csv")
aus_df = pd.DataFrame(aus_articles)
guardian_raw = pd.concat([existing, aus_df], ignore_index=True)
guardian_raw.to_csv("../data/guardian_raw.csv", index=False)
print(f"Total after Australia: {len(guardian_raw)} articles")
print(f"Countries: {guardian_raw['country'].nunique()}")

Total after Australia: 239282 articles
Countries: 196


In [28]:
uk_articles = fetch_articles_slow("United Kingdom", "section:uk-news")
print(f"UK: {len(uk_articles)} articles")

  ...page 50, 10000 articles so far
  ...page 100, 20000 articles so far
  ...page 150, 30000 articles so far
  Rate limited — switching to key 1
UK: 37296 articles


In [31]:
existing = pd.read_csv("../data/guardian_raw.csv")
uk_df = pd.DataFrame(uk_articles)
guardian_raw = pd.concat([existing, uk_df], ignore_index=True)
guardian_raw.to_csv("../data/guardian_raw.csv", index=False)
print(f"Total after UK: {len(guardian_raw)} articles")

Total after UK: 276578 articles


## World Bank Income Classification

In [59]:
OWID_URL = (
    "https://ourworldindata.org/grapher/world-bank-income-groups.csv"
    "?v=1&csvType=full&useColumnShortNames=false"
)

wb_raw = pd.read_csv(
    OWID_URL,
    storage_options={"User-Agent": "Our World In Data data fetch/1.0"}
)

print("=== World Bank Income Classification ===")
print(f"Shape: {wb_raw.shape}")
print(f"Columns: {list(wb_raw.columns)}")
print(f"Years available: {wb_raw['Year'].min()} – {wb_raw['Year'].max()}")

# Keep only the most recent year — we use a single static classification
# per country rather than a time-varying one, which is sufficient for
# our binary Global South / Global North label.
wb_latest = (
    wb_raw
    .sort_values("Year", ascending=False)
    .drop_duplicates(subset="Entity")
    .rename(columns={"Entity": "country", "Year": "wb_year"})
)

# Save locally so the notebook is reproducible without an internet connection
wb_latest.to_csv("../data/worldbank_income.csv", index=False)
print(f"\nSaved to data/worldbank_income.csv")
print(f"Countries classified: {wb_latest['country'].nunique()}")
wb_latest.head()


=== World Bank Income Classification ===
Shape: (7953, 4)
Columns: ['Entity', 'Code', 'Year', "World Bank's income classification"]
Years available: 1987 – 2024

Saved to data/worldbank_income.csv
Countries classified: 226


,country,Code,wb_year,World Bank's income classification
4148,Liberia,LBR,2024,Low-income countries
7914,Zambia,ZMB,2024,Lower-middle-income countries
37,Afghanistan,AFG,2024,Low-income countries
4110,Lesotho,LSO,2024,Lower-middle-income countries
4186,Libya,LBY,2024,Upper-middle-income countries


In [23]:
INCOME_COL = [c for c in wb_latest.columns if "income" in c.lower()][0]
print(f"Using income column: '{INCOME_COL}'")

# Apply binary label
wb_latest["region_group"] = wb_latest[INCOME_COL].apply(
    lambda x: "Global North" if str(x).strip() == "High-income countries" else "Global South"
)

# --- Manual overrides for edge cases ---
# Some countries are misclassified by income group alone for our purposes.
# We document each override explicitly with a reason.
OVERRIDES = {
    # Geopolitically Global North despite upper-middle income classification
    "Ukraine":      "Global North",
    # High income but geopolitically and geographically Global South
    "Saudi Arabia": "Global South",
    "Qatar":        "Global South",
    "Kuwait":       "Global South",
    "Bahrain":      "Global South",
    "Oman":         "Global South",
    "United Arab Emirates": "Global South",
}

for country, label in OVERRIDES.items():
    mask = wb_latest["country"] == country
    if mask.any():
        wb_latest.loc[mask, "region_group"] = label
        print(f"  Override: {country} -> {label}")

# --- Build the two lists ---
# We classify all World Bank countries here without filtering to ACLED yet.
# Name harmonisation between World Bank and ACLED happens in Section 3.
# Filtering to conflict-affected countries only will be applied post-merge.

global_north_countries = sorted(
    wb_latest[wb_latest["region_group"] == "Global North"]["country"].tolist()
)

global_south_countries = sorted(
    wb_latest[wb_latest["region_group"] == "Global South"]["country"].tolist()
)

print(f"Global North countries in ACLED:      {len(global_north_countries)}")
print(f"Global South countries in ACLED: {len(global_south_countries)}")
print(f"Global North sample:      {global_north_countries[:5]}")
print(f"Global South sample: {global_south_countries[:5]}")

# Save the classification back to CSV for use in Section 3
wb_latest.to_csv("../data/worldbank_income.csv", index=False)
print("\nUpdated worldbank_income.csv with region_group column")

Using income column: 'World Bank's income classification'
  Override: Ukraine -> Global North
  Override: Saudi Arabia -> Global South
  Override: Qatar -> Global South
  Override: Kuwait -> Global South
  Override: Bahrain -> Global South
  Override: Oman -> Global South
  Override: United Arab Emirates -> Global South
Global North countries in ACLED:      85
Global South countries in ACLED: 141
Global North sample:      ['American Samoa', 'Andorra', 'Antigua and Barbuda', 'Aruba', 'Australia']
Global South sample: ['Afghanistan', 'Albania', 'Algeria', 'Angola', 'Argentina']

Updated worldbank_income.csv with region_group column


# 3. Preparation of the Data for Analysis

# IDA for guardian_raw file

## 1. Basic structure

In [60]:
print(guardian_raw.shape)
print(guardian_raw.dtypes)
print(guardian_raw.head())

(276578, 7)
country      object
date         object
section      object
wordcount    object
title        object
pillar       object
year          int32
dtype: object
       country                  date             section wordcount  \
0  Afghanistan  2024-12-31T18:41:27Z             UK news       495   
1  Afghanistan  2024-12-29T17:09:11Z               Media       471   
2  Afghanistan  2024-12-23T17:00:21Z             US news      2008   
3  Afghanistan  2024-12-06T17:50:37Z  Global development       446   
4  Afghanistan  2024-12-06T10:40:29Z               Sport       471   

                                               title pillar  year  
0  Special forces troops face prosecution over al...   News  2024  
1                                David Page obituary   News  2024  
2  ‘Never underestimate what neighbors can do for...   News  2024  
3  Taliban move to ban women training as nurses a...   News  2024  
4  Top Afghan cricketers urge Taliban to reverse ...  Sport  2024  


## 2. Missing values

In [33]:
print(guardian_raw.isnull().sum())
print(f"Wordcount missing: {guardian_raw['wordcount'].isnull().sum()} ({guardian_raw['wordcount'].isnull().mean()*100:.1f}%)")

country         0
date            0
section         0
wordcount       0
title           0
pillar       1101
dtype: int64
Wordcount missing: 0 (0.0%)


1101 pillar's missing is fine as less than 0.5% is missing (1101/276000)

## 3. Duplicates

In [34]:
print(f"Duplicate rows: {guardian_raw.duplicated().sum()}")
print(f"Duplicate titles: {guardian_raw.duplicated(subset='title').sum()}")

Duplicate rows: 0
Duplicate titles: 50496


The duplicates is fine as guardian uses generic title names such as "Ukraine war - Latest updates" or "News roundup"

## 4. Wordcount type check — it comes in as string, needs converting

In [35]:
print(guardian_raw['wordcount'].dtype)
print(guardian_raw['wordcount'].sample(10))

object
27685     3697
151216    1033
272039     547
96329      546
29563      927
167153    1606
99193      470
58334      788
67518      768
10107      565
Name: wordcount, dtype: object


## 5. Section distribution — are there unexpected sections?

In [36]:
print(guardian_raw['section'].value_counts().head(20))

section
World news            84890
US news               43716
Australia news        42085
UK news               41031
Opinion               13245
Global development     8103
Business               5973
Environment            5564
Politics               4502
Film                   3848
Books                  2325
Art and design         1915
Media                  1656
Music                  1564
Technology             1482
Sport                  1481
News                   1479
Cities                 1442
Football               1392
Science                1067
Name: count, dtype: int64


## 6. Date format check

In [37]:
print(guardian_raw['date'].sample(5))

46273     2022-11-18T12:00:17Z
83018     2018-09-06T12:10:54Z
254845    2021-01-04T22:12:37Z
215723    2022-06-21T20:00:10Z
10943     2017-05-16T01:26:21Z
Name: date, dtype: object


## 7. Articles per country — any countries with suspiciously low counts?

In [38]:
print(guardian_raw['country'].value_counts().tail(20))

country
Dominica                            26
Seychelles                          24
eSwatini                            23
Reunion                             23
Equatorial Guinea                   22
Togo                                22
Djibouti                            21
Saint Vincent and the Grenadines    20
Saint Kitts and Nevis               19
Turks and Caicos Islands            17
Suriname                            17
Cape Verde                          13
Guinea-Bissau                       13
Mayotte                             11
Montserrat                          10
San Marino                           9
Liechtenstein                        7
Andorra                              5
Comoros                              5
Sao Tome and Principe                5
Name: count, dtype: int64


## 8. Year distribution — any gaps?

In [39]:
guardian_raw['year'] = pd.to_datetime(guardian_raw['date']).dt.year
print(guardian_raw['year'].value_counts().sort_index())

year
2015    24888
2016    21280
2017    24875
2018    25527
2019    24802
2020    26265
2021    26355
2022    35128
2023    33595
2024    33863
Name: count, dtype: int64


A clear jump in articles written in 2022 (35128 articles) from the previous year (26355 articles) certainly in part motivated by the war in Ukraine.

# IDA for ACLED file (cleaned_fatalities)

## 1. Basic structure

In [61]:
print(acled.shape)
print(acled.dtypes)
print(acled.head())

(1634, 4)
COUNTRY         object
YEAR             int64
FATALITIES       int64
region_group    object
dtype: object
       COUNTRY  YEAR  FATALITIES  region_group
0  Afghanistan  2017       36360  Global South
1  Afghanistan  2018       42991  Global South
2  Afghanistan  2019       41419  Global South
3  Afghanistan  2020       31037  Global South
4  Afghanistan  2021       42425  Global South


## 2. Missing values

In [62]:
print(acled.isnull().sum())

COUNTRY         0
YEAR            0
FATALITIES      0
region_group    0
dtype: int64


## 3. Year range

In [63]:
print(acled['YEAR'].value_counts().sort_index())

YEAR
2015     61
2016     74
2017     77
2018    161
2019    161
2020    212
2021    245
2022    215
2023    213
2024    215
Name: count, dtype: int64


## 4. Fatalities distribution — any zeroes or negatives?

In [65]:
print(acled['FATALITIES'].describe())
print(f"Zero fatality rows: {(acled['FATALITIES'] == 0).sum()}")

count     1634.000000
mean       991.755202
std       4549.874372
min          0.000000
25%          0.000000
50%          4.000000
75%        150.000000
max      73573.000000
Name: FATALITIES, dtype: float64
Zero fatality rows: 566


## 5. Duplicates

In [66]:
print(f"Duplicate country-year pairs: {acled.duplicated(subset=['COUNTRY','YEAR']).sum()}")

Duplicate country-year pairs: 0


# IDA for World Bank income data 

## 1. Basic structure

In [69]:
wb_latest = wb_latest.reset_index(drop=True)
print(wb_latest.shape)
print(wb_latest[['country', INCOME_COL]].head(20))

(226, 4)
          country World Bank's income classification
0         Liberia               Low-income countries
1          Zambia      Lower-middle-income countries
2     Afghanistan               Low-income countries
3         Lesotho      Lower-middle-income countries
4           Libya      Upper-middle-income countries
5         Lebanon      Lower-middle-income countries
6   Liechtenstein              High-income countries
7          Latvia              High-income countries
8            Laos      Lower-middle-income countries
9       Lithuania              High-income countries
10     Kyrgyzstan      Lower-middle-income countries
11     Luxembourg              High-income countries
12         Kuwait              High-income countries
13          Macao              High-income countries
14         Kosovo      Upper-middle-income countries
15       Kiribati      Lower-middle-income countries
16     Madagascar               Low-income countries
17          Kenya      Lower-middle-i

## 2. How many in each group?

In [73]:
print(wb_latest[INCOME_COL].value_counts())

World Bank's income classification
High-income countries            90
Upper-middle-income countries    59
Lower-middle-income countries    51
Low-income countries             26
Name: count, dtype: int64


## 3. Any nulls in income column?

In [74]:
print(wb_latest[INCOME_COL].isnull().sum())

0


## 4. Countries in ACLED missing from World Bank classification

In [80]:
acled_names = set(acled['COUNTRY'].unique())
wb_names = set(wb_latest['Entity'].unique())
print("In ACLED but not World Bank:")
print(sorted(acled_names - wb_names))

In ACLED but not World Bank:
['Akrotiri and Dhekelia', 'Anguilla', 'Antarctica', 'Bailiwick of Guernsey', 'Bailiwick of Jersey', 'British Indian Ocean Territory', 'Caribbean Netherlands', 'Christmas Island', 'Cocos (Keeling) Islands', 'Cook Islands', 'Falkland Islands', 'French Southern and Antarctic Lands', 'Guadeloupe', 'Heard Island and McDonald Islands', 'Ivory Coast', 'Martinique', 'Micronesia', 'Montserrat', 'Niue', 'Norfolk Island', 'Pitcairn', 'Reunion', 'Saint Helena, Ascension and Tristan da Cunha', 'Saint Pierre and Miquelon', 'Saint-Barthelemy', 'Saint-Martin', 'Sint Maarten', 'South Georgia and the South Sandwich Islands', 'Tokelau', 'United States Minor Outlying Islands', 'Vatican City', 'Virgin Islands, U.S.', 'Wallis and Futuna']


# Cleaning Process

In [50]:
# =============================================================================
# SECTION 3.4 — DATA CLEANING
# =============================================================================
# Based on IDA findings, we perform the following cleaning steps:
# 1. Convert date column to datetime
# 2. Convert wordcount column to numeric
# 3. Handle missing pillar values
# 4. Remove duplicate titles
# 5. Filter ACLED to study window (already done in friend's notebook)
# 6. Standardise column names across datasets
# =============================================================================

# =============================================================================
# 3.4.1 CLEAN GUARDIAN DATA
# =============================================================================

guardian_clean = guardian_raw.copy()

# --- Convert date to datetime ---
# IDA showed dates are valid ISO 8601 strings (e.g. 2022-11-18T12:00:17Z)
# We convert to datetime and extract year and month for time series analysis
guardian_clean['date'] = pd.to_datetime(guardian_clean['date'], utc=True)
guardian_clean['year']  = guardian_clean['date'].dt.year
guardian_clean['month'] = guardian_clean['date'].dt.month

print("Date conversion:")
print(guardian_clean['date'].dtype)
print(guardian_clean[['date', 'year', 'month']].head())

# --- Convert wordcount to numeric ---
# IDA showed wordcount is stored as string (object dtype)
# errors='coerce' turns any non-numeric values into NaN rather than crashing
guardian_clean['wordcount'] = pd.to_numeric(
    guardian_clean['wordcount'], errors='coerce'
)

print(f"\nWordcount dtype after conversion: {guardian_clean['wordcount'].dtype}")
print(f"Wordcount NaNs after conversion:  {guardian_clean['wordcount'].isnull().sum()}")

# --- Handle missing pillar values ---
# IDA found 1,101 missing pillar values (~0.4% of rows)
# These are minor and likely live blogs or special content formats
# We fill with 'Unknown' to retain these rows for volume analysis
guardian_clean['pillar'] = guardian_clean['pillar'].fillna('Unknown')

print(f"\nPillar nulls after fill: {guardian_clean['pillar'].isnull().sum()}")

# --- Handle duplicate titles ---
# IDA found 50,496 duplicate titles — these are articles tagged with
# multiple countries, not true duplicates. We retain all rows as each
# country-article pair is a valid observation for our analysis.
# We document this decision explicitly rather than dropping them.
print(f"\nDuplicate titles retained: {guardian_clean.duplicated(subset='title').sum()}")
print("Note: duplicates are articles tagged with multiple countries — retained intentionally")

# --- Final structure check ---
print(f"\nGuardian clean shape: {guardian_clean.shape}")
print(guardian_clean.dtypes)

# =============================================================================
# 3.4.2 CLEAN ACLED DATA
# =============================================================================

acled_clean = acled.copy()

# --- Standardise column names to lowercase ---
acled_clean.columns = acled_clean.columns.str.lower()

# --- Confirm year filter applied (2015-2024) ---
# This was handled in the data preparation notebook but we verify here
acled_clean = acled_clean[
    (acled_clean['year'] >= 2015) &
    (acled_clean['year'] <= 2024)
]

print(f"\nACLED clean shape: {acled_clean.shape}")
print(f"Year range: {acled_clean['year'].min()} – {acled_clean['year'].max()}")

# --- Check zero fatality rows ---
# IDA found 927 rows with zero fatalities (31.5% of dataset)
# These represent country-years with recorded conflict but no deaths
# We retain them — zero fatalities is a valid severity value
zero_fat = (acled_clean['fatalities'] == 0).sum()
print(f"Zero fatality rows retained: {zero_fat}")

# =============================================================================
# 3.4.3 CLEAN WORLD BANK DATA
# =============================================================================

wb_clean = wb_latest.copy()

# --- Standardise column names ---
wb_clean = wb_clean.rename(columns={
    "Entity":       "country",
    "World Bank's income classification": "income_group"
})

# --- Keep only relevant columns ---
wb_clean = wb_clean[['country', 'income_group', 'region_group']]

# --- Confirm no nulls in key columns ---
print(f"\nWorld Bank nulls:")
print(wb_clean.isnull().sum())
print(f"\nRegion group counts:")
print(wb_clean['region_group'].value_counts())

# =============================================================================
# 3.4.4 SUMMARY
# =============================================================================

print("\n=== Cleaning Summary ===")
print(f"Guardian: {len(guardian_raw)} -> {len(guardian_clean)} rows")
print(f"ACLED:    {len(acled)} -> {len(acled_clean)} rows")
print(f"World Bank: {len(wb_latest)} -> {len(wb_clean)} rows")

Date conversion:
datetime64[ns, UTC]
                       date  year  month
0 2024-12-31 18:41:27+00:00  2024     12
1 2024-12-29 17:09:11+00:00  2024     12
2 2024-12-23 17:00:21+00:00  2024     12
3 2024-12-06 17:50:37+00:00  2024     12
4 2024-12-06 10:40:29+00:00  2024     12

Wordcount dtype after conversion: int64
Wordcount NaNs after conversion:  0

Pillar nulls after fill: 0

Duplicate titles retained: 50496
Note: duplicates are articles tagged with multiple countries — retained intentionally

Guardian clean shape: (276578, 8)
country                   object
date         datetime64[ns, UTC]
section                   object
wordcount                  int64
title                     object
pillar                    object
year                       int32
month                      int32
dtype: object

ACLED clean shape: (1634, 3)
Year range: 2015 – 2024
Zero fatality rows retained: 566

World Bank nulls:
country         0
income_group    0
region_group    0
dtype: int64

Regio

# 4. Exploratory Data Analysis

# 5. Conclusion